# Accuracy analysis

Loads `summary.json` files produced by the `batch` command of
`sfm_yolo.src.main` and compares the predicted depths against
ground-truth measurements supplied in a CSV.

Expected ground-truth schema (`data/ground_truth.csv`):

    clip_id,depth_m
    0001,0.045
    0002,0.090
    ...

If you do not have ground truth yet, the second cell still gives you
**summary statistics across the whole batch** (mean depth, confidence
histogram, success rate).

In [ ]:
import sys, json, pathlib
ROOT = pathlib.Path('..').resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sfm_yolo.src.utils.evaluation_metrics import compute_depth_metrics

BATCH_DIR = ROOT / 'sfm_yolo' / 'outputs' / 'runs' / 'test_batch'
INDEX = BATCH_DIR / '_index.json'
print('Looking at', INDEX, '->', INDEX.exists())

In [ ]:
rows = []
if INDEX.exists():
    data = json.loads(INDEX.read_text())
    for s in data.get('summaries', []):
        clip = pathlib.Path(s['video']).stem
        for tr in s.get('tracks', []):
            rows.append({
                'clip_id': clip,
                'track_id': tr['track_id'],
                'depth_m': tr['result']['depth_m'],
                'confidence': tr['result']['confidence'],
                'method': tr['result']['method'],
                'sfm_depth_m': tr['result'].get('sfm_depth_m'),
            })
df = pd.DataFrame(rows)
print(df.head())
print('Confirmed pothole estimates:', len(df))

In [ ]:
if len(df):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].hist(df['depth_m'].dropna() * 100, bins=30)
    ax[0].set_xlabel('predicted depth (cm)'); ax[0].set_ylabel('count')
    ax[0].set_title('Depth distribution (all tracks)')
    ax[1].hist(df['confidence'].dropna(), bins=20)
    ax[1].set_xlabel('confidence'); ax[1].set_ylabel('count'); ax[1].set_title('Confidence histogram')
    plt.tight_layout(); plt.show()

In [ ]:
GT_PATH = ROOT / 'data' / 'ground_truth.csv'
if GT_PATH.exists() and len(df):
    gt = pd.read_csv(GT_PATH)
    merged = df.merge(gt, on='clip_id', how='inner', suffixes=('', '_gt'))
    merged = merged.rename(columns={'depth_m_gt': 'truth_m'})
    # Use the highest-confidence track per clip
    best = merged.sort_values('confidence', ascending=False).groupby('clip_id').head(1)
    metrics = compute_depth_metrics(best['depth_m'].to_numpy(), best['truth_m'].to_numpy())
    print(metrics.as_dict())

    plt.figure(figsize=(5, 5))
    plt.scatter(best['truth_m'] * 100, best['depth_m'] * 100)
    lim = max(best['truth_m'].max(), best['depth_m'].max()) * 110
    plt.plot([0, lim], [0, lim], 'k--', alpha=0.5)
    plt.plot([0, lim], [0, lim * 1.05], 'g:', label='+5%')
    plt.plot([0, lim], [0, lim * 0.95], 'g:', label='-5%')
    plt.xlabel('ground truth (cm)'); plt.ylabel('predicted (cm)')
    plt.title('Predicted vs ground-truth pothole depth')
    plt.legend(); plt.grid(True); plt.show()
else:
    print('No ground_truth.csv found - skipping accuracy comparison.')